# Week 2 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-02.md`](../../weeks/week-02.md) · **Given engine:** [`otp.py`](otp.py)

Teaching walkthrough of the week-2 studio. It **imports the reference answers from
[`solution.py`](solution.py)** — never re-pasting them — so what runs here is exactly
what `test_otp.py` grades. Correctness is verified separately:
`python3 studios/_verify_solutions.py week-02`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: week folder (for solution/otp) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-02"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import otp
import solution
c1, c2 = otp.load_ciphertext_pair()   # the intercepted two-time-pad: no key, no plaintext

## Task 1 — entropy and unicity: *why* week 1's break was inevitable

`unicity_for_substitution` is pure measurement over the **public** engine: key
entropy `H(K) = log2(26!)` and Shannon's unicity distance `U = H(K)/D`, where `D`
is English redundancy. The number `U ≈ 27.6` characters is the whole point — week
1's message was hundreds of chars, *far* past `U`, so exactly one key fit. Below
~28 chars the break would have been ambiguous. Entropy is necessary, not
sufficient; redundancy is the other half.

In [ ]:
print(inspect.getsource(solution.unicity_for_substitution))
H_K, U = solution.unicity_for_substitution()
# mirror test_unicity_distance_explains_week1
assert 88.0 < H_K < 89.0, H_K
assert 27.0 < U < 28.5, U
print(f"H(K) = {H_K:.1f} bits, unicity U = {U:.1f} chars")
print("substitution has MORE key entropy than 56-bit DES, yet week 1 broke it in a second")

## Task 2 — the one-time pad: perfect secrecy made concrete

`key_that_decrypts_to` is the constructive core of Shannon's proof: for **any**
decoy of the right length a key *exists* (`key = ct ⊕ decoy`) that makes the SAME
ciphertext decrypt to it. So one ciphertext yields two *different meaningful*
messages under two keys — it favours neither. That is a guarantee **with no
condition on the attacker's compute**: the strongest in the course.

In [ ]:
print(inspect.getsource(solution.key_that_decrypts_to))
# mirror test_otp_perfect_secrecy_when_key_used_once
msg   = b"ATTACK AT DAWN"
decoy = b"RETREAT NOW!!!"          # same length, opposite meaning
key   = bytes((i * 37 + 11) % 256 for i in range(len(msg)))
ct    = otp.xor(msg, key)
assert otp.xor(ct, key) == msg
key2 = solution.key_that_decrypts_to(ct, decoy)
assert otp.xor(ct, key2) == decoy and key2 != key
print("one ciphertext ->", repr(otp.xor(ct, key)), "OR", repr(otp.xor(ct, key2)))
print("same bytes, two meaningful messages: the ciphertext cannot betray the real one")

## Task 3 — the break: reuse the key ONCE and the guarantee collapses

Perfect secrecy has exactly one condition — the key is used *once*. Reuse it and
the key cancels: `c1 ⊕ c2 = (p1⊕K) ⊕ (p2⊕K) = p1 ⊕ p2`. The attacker holds two
ciphertexts and never needed the key. `crib_drag` slides a guessed common word
across `x = c1 ⊕ c2`; where the crib sits at its true spot in one message, the
*other* message's text surfaces. `recover_other_plaintext` then peels off the
whole second message from a full guess of the first.

In [ ]:
print(inspect.getsource(solution.crib_drag))
print(inspect.getsource(solution.recover_other_plaintext))

### Watch the guarantee fail — live

This reproduces `test_two_time_pad_leaks_and_crib_drag_recovers`. The crib
`please` hits position 0 and reveals `p1[0:6]`; then a full plaintext recovers the
other entirely. **Control Scorecard terms:** perfect secrecy is a *GUARANTEE* —
but *conditional* on single use. Reuse violates the condition and the guarantee
does not degrade gracefully; it collapses to total plaintext recovery.

In [ ]:
P1 = b"the launch code is four seven two the target is the north bridge tonight"[:72]
P2 = b"please water my plants and feed the cat while i am away for the weekend ok"[:72]
x = otp.xor(c1, c2)
assert x == otp.xor(P1, P2), "the key cancels: c1 XOR c2 == p1 XOR p2"

please_hits = dict(solution.crib_drag(x, b"please"))
assert please_hits.get(0) == P1[0:6], please_hits.get(0)
assert dict(solution.crib_drag(x, b"target")), "crib 'target' should surface a fragment"
print(f"crib 'please' hit position 0 -> revealed {please_hits[0]!r} (start of message 1)")

rec2 = solution.recover_other_plaintext(c1, c2, P1)
rec1 = solution.recover_other_plaintext(c2, c1, P2)
assert rec2 == P2 and rec1 == P1
print("recovered p2:", rec2.decode())
print("recovered p1:", rec1.decode())
print()
print("LESSON (scorecard axis 2): perfect secrecy is a GUARANTEE against unbounded")
print("compute -- CONDITIONAL on using the key exactly once. Reuse => total collapse,")
print("not graceful degradation. Naming the condition is naming the attack.")